In [ ]:
%%capture
# Core imports
import json
import csv
import re
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Set, Tuple
from datetime import datetime
from collections import Counter
from html import escape
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# Jupyter widgets
import ipywidgets as widgets
from ipywidgets import Layout, Button, Box, VBox, HBox, Label, HTML as HTMLWidget

# Progress tracking
from tqdm.auto import tqdm

# Country lookup
from _library.countries import CountryLookup

# Optional: Fuzzy matching (for keyword search)
try:
    from fuzzywuzzy import fuzz
    FUZZY_AVAILABLE = True
except ImportError:
    FUZZY_AVAILABLE = False

print('✅ Imports loaded successfully')
if FUZZY_AVAILABLE:
    print('✅ Fuzzy matching available')
else:
    print('⚠️  Fuzzy matching not available (install fuzzywuzzy for fuzzy keyword matching)')
print(f'📅 Initialized: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

In [ ]:
%%capture
# Define paths
BASE_PATH = Path('NSDDD_v3_workspace/metadata')
MODEL_PATH = Path('NSDDD_v3_workspace/model')

# Metadata dictionary to store all loaded metadata
METADATA = {}

def load_organizational_membership(filename: str) -> Set[str]:
    """Load organizational membership and return set of ISO Alpha-3 codes."""
    filepath = BASE_PATH / filename
    members = set()

    if not filepath.exists():
        print(f'⚠️  {filename} not found')
        return members

    try:
        # Special handling for G77.csv (has duplicate Mexico entries)
        if filename == 'G77.csv':
            with open(filepath, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                country_status = {}

                for row in reader:
                    country_code = row.get('ISO-Alpha3 Code', '').strip()
                    is_member = row.get('Member', '').strip()
                    is_former = row.get('Former', '').strip()

                    # Track current status - prioritize most recent entry
                    if is_member and not is_former:
                        country_status[country_code] = True
                    elif is_former:
                        country_status[country_code] = False

                members = {code for code, is_current in country_status.items() if is_current}
                return members

        # Standard CSV handling for other files
        df = pd.read_csv(filepath)

        # Find the ISO code column (handle various naming conventions)
        iso_col = None
        for col in df.columns:
            if 'iso' in col.lower() and ('alpha' in col.lower() or 'code' in col.lower()):
                iso_col = col
                break

        if iso_col is None:
            print(f'⚠️  No ISO code column found in {filename}')
            return set()

        # Filter for current members (no year left or year left is empty)
        if 'Year left' in df.columns:
            # Robust check: handle numeric, string, or mixed columns
            col = df['Year left']
            current_members = df[col.isna() | (col.astype(str).str.strip() == '')]
        else:
            current_members = df

        members = set(current_members[iso_col].dropna().str.strip())
        return members
    except Exception as e:
        print(f'⚠️  Error loading {filename}: {e}')
        return set()

# Load organizational memberships
print('\n🔄 Loading organizational metadata...')
organizations = {
    'NATO': load_organizational_membership('NATO.csv'),
    'EU': load_organizational_membership('EU.csv'),
    'ASEAN': load_organizational_membership('ASEAN.csv'),
    'African Union': load_organizational_membership('AU.csv'),
    'BRICS': load_organizational_membership('BRICS.csv'),
    'Commonwealth': load_organizational_membership('Commonwealth.csv'),
    'G7': load_organizational_membership('G7.csv'),
    'G20': load_organizational_membership('G20.csv'),
    'G77': load_organizational_membership('G77.csv'),
    'OECD': load_organizational_membership('OECD.csv'),
    'Arab League': load_organizational_membership('Arab_League.csv'),
    'GCC': load_organizational_membership('GCC_Gulf_Cooperation_Council.csv'),
    'OAS': load_organizational_membership('OAS.csv'),
    'OSCE': load_organizational_membership('OSCE.csv'),
    'SCO': load_organizational_membership('SCO.csv'),
    'CSTO': load_organizational_membership('CSTO.csv'),
    'CIS': load_organizational_membership('CIS.csv'),
    'SIDS': load_organizational_membership('SIDS.csv'),
    'LDCs': load_organizational_membership('LDCs_Least_Developed_Countries.csv'),
    'NAM': load_organizational_membership('NAM_Non_Aligned_Movement.csv'),
    'P5': load_organizational_membership('P5_UN_Security_Council.csv'),
    'Francophonie': load_organizational_membership('Francophonie.csv'),
    'ANZUS': load_organizational_membership('ANZUS.csv'),
    'AUKUS': load_organizational_membership('AUKUS.csv'),
    'UFM': load_organizational_membership('UFM.csv'),
    'DCAF': load_organizational_membership('DCAF.csv'),
}

METADATA['organizations'] = organizations

for org, members in organizations.items():
    if members:
        print(f'  ✓ {org}: {len(members)} members')

# Load UN regional codes
print('\n🔄 Loading UN regional classifications...')
try:
    iso_df = pd.read_csv(BASE_PATH / 'ISO-3166-Countries-with-Regional-Codes.csv')

    # Create region mappings
    METADATA['regions'] = dict(zip(iso_df['alpha-3'], iso_df['region']))
    METADATA['subregions'] = dict(zip(iso_df['alpha-3'], iso_df['sub-region']))
    METADATA['intermediate_regions'] = dict(zip(iso_df['alpha-3'], iso_df['intermediate-region']))

    # Get unique regions and subregions
    unique_regions = sorted(iso_df['region'].dropna().unique())
    unique_subregions = sorted(iso_df['sub-region'].dropna().unique())

    print(f'  ✓ Regions: {len(unique_regions)} ({", ".join(unique_regions)})')
    print(f'  ✓ Subregions: {len(unique_subregions)}')

    METADATA['unique_regions'] = unique_regions
    METADATA['unique_subregions'] = unique_subregions

except Exception as e:
    print(f'⚠️  Error loading regional data: {e}')

# Load Freedom House data
print('\n🔄 Loading Freedom House democracy ratings...')
try:
    fh_df = pd.read_csv(BASE_PATH / 'Freedom_house.csv')

    # Freedom House uses columns for status, with years as values
    # Find the ISO column
    iso_col = [c for c in fh_df.columns if 'iso' in c.lower()][0]

    # Reshape to get most recent status
    freedom_status = {}
    for _, row in fh_df.iterrows():
        country_code = row[iso_col]
        # Check which status column has a value
        if pd.notna(row.get('Free')):
            freedom_status[country_code] = 'Free'
        elif pd.notna(row.get('Partly Free')):
            freedom_status[country_code] = 'Partly Free'
        elif pd.notna(row.get('Not Free')):
            freedom_status[country_code] = 'Not Free'

    METADATA['freedom_house'] = freedom_status

    # Count by status
    status_counts = Counter(freedom_status.values())
    for status, count in status_counts.items():
        print(f'  ✓ {status}: {count} countries')

except Exception as e:
    print(f'⚠️  Error loading Freedom House data: {e}')

# Load ODA data
print('\n🔄 Loading ODA recipient status...')
try:
    oda_df = pd.read_csv(BASE_PATH / 'ODA.csv')
    # Get latest status for each country
    oda_latest = oda_df.sort_values('Year', ascending=False).groupby('ISO_Code').first().reset_index()
    oda_mapping = dict(zip(oda_latest['ISO_Code'], oda_latest['Status']))
    METADATA['oda_mapping'] = oda_mapping
    print(f'  ✓ ODA data loaded ({len(oda_latest)} countries with status)')

    # Count countries by status
    oda_counts = oda_latest['Status'].value_counts()
    for status, count in oda_counts.items():
        print(f'  ✓ {status}: {count} countries')

except Exception as e:
    print(f'⚠️  Error loading ODA data: {e}')
    oda_mapping = {}
    METADATA['oda_mapping'] = oda_mapping

# Load World Bank income classifications
print('\n🔄 Loading World Bank income classifications...')
try:
    # OGHIST.csv has complex header structure - skip first 11 rows
    wb_df = pd.read_csv(BASE_PATH / 'OGHIST.csv', skiprows=11, header=None)

    # Columns: 0=ISO3, 1=Country, 2+=Classifications by year
    wb_df.columns = ['ISO3', 'Country'] + [f'Year_{i}' for i in range(len(wb_df.columns) - 2)]

    # Get most recent non-empty classification for each country
    income_groups = {}
    for _, row in wb_df.iterrows():
        iso_code = row['ISO3']
        # Get last non-empty classification
        classifications = [row[col] for col in wb_df.columns[2:] if pd.notna(row[col]) and row[col] not in ['..', '']]
        if classifications:
            last_class = classifications[-1]
            # Map abbreviations to full names
            class_map = {'L': 'Low', 'LM': 'Lower-middle', 'UM': 'Upper-middle', 'H': 'High'}
            income_groups[iso_code] = class_map.get(last_class, last_class)

    METADATA['income_groups'] = income_groups

    # Count by income group
    income_counts = Counter(income_groups.values())
    for income, count in sorted(income_counts.items()):
        print(f'  ✓ {income}: {count} countries')

except Exception as e:
    print(f'⚠️  Error loading World Bank data: {e}')

# Load document metadata
print('\n🔄 Loading document metadata...')
try:
    doc_df = pd.read_csv(BASE_PATH / 'document_metadata.csv')
    METADATA['documents'] = doc_df
    print(f'  ✓ Loaded {len(doc_df)} documents')
    print(f'  ✓ Countries: {doc_df["ISO Alpha-3 Code"].nunique()}')
    print(f'  ✓ Year range: {doc_df["Year"].min()}–{doc_df["Year"].max()}')
    print(f'  ✓ Document types: {", ".join(doc_df["NSS/WP/DD"].unique())}')

except Exception as e:
    print(f'⚠️  Error loading document metadata: {e}')


# Load Small States classifications
print('\n🔄 Loading Small States classifications...')
try:
    small_states_df = pd.read_csv(BASE_PATH / 'Small_States.csv')

    # Get ISO code column
    iso_col = 'ISO-Alpha3 Code'

    # Create sets for each classification
    small_states = set(small_states_df[small_states_df['Small_States'] == 'Yes'][iso_col].dropna().str.strip())
    ssf_members = set(small_states_df[small_states_df['SSF_Member'] == 'Yes'][iso_col].dropna().str.strip())
    un_sids_members = set(small_states_df[small_states_df['UN_SIDS_Member'] == 'Yes'][iso_col].dropna().str.strip())

    METADATA['small_states'] = small_states
    METADATA['ssf_members'] = ssf_members
    METADATA['un_sids_members'] = un_sids_members

    print(f'  ✓ Small States: {len(small_states)} countries')
    print(f'  ✓ Small States Forum (SSF): {len(ssf_members)} countries')
    print(f'  ✓ UN SIDS: {len(un_sids_members)} countries')

except Exception as e:
    print(f'⚠️  Error loading Small States data: {e}')

# Load Huntington's civilizational classifications
print('\n🔄 Loading Huntington civilizational classifications...')
try:
    hunt_df = pd.read_csv(BASE_PATH / 'Huntington_classification.csv')

    # Create mapping: ISO code -> classification
    huntington_map = dict(zip(hunt_df['ISO-Alpha3 Code'].str.strip(),
                             hunt_df['Huntingdons_class'].str.strip()))

    METADATA['huntington'] = huntington_map

    # Get unique classifications
    unique_civs = sorted(hunt_df['Huntingdons_class'].dropna().unique())
    METADATA['huntington_civilizations'] = unique_civs

    civ_counts = Counter(huntington_map.values())
    print(f'  ✓ Loaded {len(huntington_map)} country classifications')
    print(f'  ✓ Civilizations: {", ".join(unique_civs[:5])}...')

except Exception as e:
    print(f'⚠️  Error loading Huntington data: {e}')

print('\n✅ All metadata loaded successfully!')
print(f'\n📊 Total metadata categories: {len(METADATA)}')

In [ ]:
%%capture
@dataclass
class MetadataFilters:
    """Comprehensive metadata filtering options."""

    # Geographic filters
    regions: List[str] = field(default_factory=list)
    subregions: List[str] = field(default_factory=list)
    countries: List[str] = field(default_factory=list)

    # Organizational filters
    organizations: List[str] = field(default_factory=list)

    # Economic/political filters
    income_groups: List[str] = field(default_factory=list)
    freedom_house_status: List[str] = field(default_factory=list)
    oda_status: List[str] = field(default_factory=list)

    # Small States and civilizational filters
    small_states_groups: List[str] = field(default_factory=list)  # 'Small States', 'SSF', 'UN SIDS'
    huntington_civilizations: List[str] = field(default_factory=list)

    # Document filters
    doc_types: List[str] = field(default_factory=list)
    year_min: Optional[int] = None
    year_max: Optional[int] = None
    most_recent_only: bool = False

    def to_dict(self) -> Dict:
        """Convert to dictionary for display."""
        return {k: v for k, v in asdict(self).items() if v not in [[], None, False]}

    def get_summary(self) -> str:
        """Get human-readable summary of active filters."""
        active = []

        if self.regions:
            active.append(f"Regions: {', '.join(self.regions)}")
        if self.subregions:
            active.append(f"Subregions: {', '.join(self.subregions[:3])}{'...' if len(self.subregions) > 3 else ''}")
        if self.countries:
            active.append(f"Countries: {', '.join(self.countries[:5])}{'...' if len(self.countries) > 5 else ''}")
        if self.organizations:
            active.append(f"Organizations: {', '.join(self.organizations)}")
        if self.income_groups:
            active.append(f"Income: {', '.join(self.income_groups)}")
        if self.freedom_house_status:
            active.append(f"Democracy: {', '.join(self.freedom_house_status)}")
        if self.oda_status:
            active.append(f"ODA: {', '.join(self.oda_status)}")
        if self.small_states_groups:
            active.append(f"Small States: {', '.join(self.small_states_groups)}")
        if self.huntington_civilizations:
            active.append(f"Civilizations: {', '.join(self.huntington_civilizations[:3])}{'...' if len(self.huntington_civilizations) > 3 else ''}")
        if self.doc_types:
            active.append(f"Doc types: {', '.join(self.doc_types)}")
        if self.year_min is not None or self.year_max is not None:
            yr_range = f"{self.year_min if self.year_min is not None else 'start'}–{self.year_max if self.year_max is not None else 'end'}"
            active.append(f"Years: {yr_range}")
        if self.most_recent_only:
            active.append("Most recent docs only")

        return ' | '.join(active) if active else 'No filters applied'

@dataclass
class SearchConfig:
    """Search configuration."""
    topics: List[str]
    search_threshold: float = 0.70
    cluster_threshold: float = 0.78
    export_prefix: str = 'metadata_search_'
    export_enabled: bool = True
    keyword_search: bool = False
    fuzzy_matching: bool = False
    fuzzy_threshold: int = 80
    smart_boundaries: bool = True  # Allow hyphenated variants
    context_window: int = 0
    max_results_per_topic: Optional[int] = None

    # Boolean search configuration
    boolean_mode: bool = False  # Enable Boolean operators (AND, OR, NOT)
    boolean_scoring: str = 'match_count'  # Scoring strategy: 'match_count', 'avg_score', or 'max_score'

    # Metadata filters
    filters: MetadataFilters = field(default_factory=MetadataFilters)

@dataclass
class SearchResult:
    """Individual search result."""
    segment_id: str
    text: str
    score: float
    document_id: str
    document_name: str
    country: str
    country_iso: str  # ISO Alpha-3 code for metadata lookups
    year: int
    doc_type: str
    context_before: str = ''
    context_after: str = ''
    cluster_id: Optional[str] = None
    cluster_name: str = ''  # TF-IDF generated cluster name

    # Metadata enrichment
    region: str = ''
    subregion: str = ''
    income_group: str = ''
    freedom_status: str = ''

print('✅ Data structures defined')

In [ ]:
%%capture
# ================================
# Section 4: Model + Search Utilities (FULL CELL)
# ================================

# Global model storage
MODEL_DICT = None
ENCODER = None

def initialize_model():
    """Initialize semantic search model."""
    global MODEL_DICT, ENCODER

    if MODEL_DICT is not None:
        print('✅ Model already loaded')
        return MODEL_DICT, ENCODER

    print('🔄 Loading model files...')

    try:
        with open(MODEL_PATH / 'documents_dict.json', 'r') as f:
            documents_dict = json.load(f)

        with open(MODEL_PATH / 'countries_dict.json', 'r') as f:
            countries_dict = json.load(f)

        with open(MODEL_PATH / 'segments_dict.json', 'r') as f:
            segments_dict = json.load(f)

        with open(MODEL_PATH / 'encoded_segments.json', 'r') as f:
            encoded_segments = json.load(f)

        print('  ✓ Loading segment encodings...')
        _npy = MODEL_PATH / 'segment_encodings.npy'
        if _npy.exists():
            segment_encodings = np.load(str(_npy))
        else:
            with open(MODEL_PATH / 'segment_encodings.json', 'r') as f:
                segment_encodings = np.array(json.load(f))

        MODEL_DICT = {
            'documents_dict': documents_dict,
            'countries_dict': countries_dict,
            'segments_dict': segments_dict,
            'encoded_segments': encoded_segments,
            'segment_encodings': segment_encodings
        }

        # Initialize encoder (optional; required for semantic search)
        try:
            from sentence_transformers import SentenceTransformer
            import logging
            import warnings

            # Suppress model loading warnings
            import os
            os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings
            os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Suppress tokenizer warnings

            warnings.filterwarnings('ignore')
            logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
            logging.getLogger('transformers').setLevel(logging.ERROR)

            # Suppress transformers model loading verbosity
            from transformers import logging as transformers_logging
            transformers_logging.set_verbosity_error()

            _local_model = MODEL_PATH / 'all-mpnet-base-v2'
            _model_path = str(_local_model) if _local_model.exists() else 'sentence-transformers/all-mpnet-base-v2'
            ENCODER = SentenceTransformer(_model_path)
            print('  ✓ Encoder initialized')
        except Exception:
            print('  ⚠️  Could not load encoder (keyword search will still work)')
            ENCODER = None

        print(f'\n✅ Model loaded successfully!')
        print(f'   📄 Documents: {len(documents_dict):,}')
        print(f'   🌍 Countries: {len(countries_dict):,}')
        print(f'   📝 Segments: {len(segments_dict):,}')
        print(f'   🔢 Encoded segments: {len(encoded_segments):,}')

        return MODEL_DICT, ENCODER

    except Exception as e:
        print(f'❌ Error loading model: {e}')
        import traceback
        traceback.print_exc()
        return None, None


def apply_metadata_filters(doc_ids: List[str], filters: 'MetadataFilters') -> List[str]:
    """
    Apply metadata filters to a list of document IDs (File prefix strings).

    Behaviour:
    - Multiple selections within a category are OR (union).
    - Selections across categories are AND (intersection).

    Returns:
        List[str] of filtered document IDs (File prefix values as strings).
    """
    if filters is None or not filters.to_dict():
        return [str(d) for d in doc_ids]

    if 'documents' not in METADATA:
        raise KeyError("METADATA['documents'] is missing. Load document_metadata.csv first.")

    doc_df = METADATA['documents']
    filtered_docs = doc_df[doc_df['File prefix'].astype(str).isin([str(d) for d in doc_ids])]

    valid_countries: Set[str] = set()

    # Direct country selection (ISO3)
    if getattr(filters, 'countries', None):
        valid_countries.update(filters.countries)

    # Regions
    if getattr(filters, 'regions', None):
        for code, region in METADATA.get('regions', {}).items():
            if region in filters.regions:
                valid_countries.add(code)

    # Subregions
    if getattr(filters, 'subregions', None):
        for code, subregion in METADATA.get('subregions', {}).items():
            if subregion in filters.subregions:
                valid_countries.add(code)

    # Organizations
    if getattr(filters, 'organizations', None):
        org_countries: Set[str] = set()
        for org in filters.organizations:
            org_countries.update(METADATA.get('organizations', {}).get(org, set()))
        valid_countries = (valid_countries & org_countries) if valid_countries else org_countries

    # Income groups
    if getattr(filters, 'income_groups', None):
        income_countries = {
            code for code, group in METADATA.get('income_groups', {}).items()
            if group in filters.income_groups
        }
        valid_countries = (valid_countries & income_countries) if valid_countries else income_countries

    # Freedom House
    if getattr(filters, 'freedom_house_status', None):
        fh_countries = {
            code for code, status in METADATA.get('freedom_house', {}).items()
            if status in filters.freedom_house_status
        }
        valid_countries = (valid_countries & fh_countries) if valid_countries else fh_countries

    # ODA status
    if getattr(filters, 'oda_status', None):
        oda_countries = {
            code for code, status in METADATA.get('oda_mapping', {}).items()
            if status in filters.oda_status
        }
        valid_countries = (valid_countries & oda_countries) if valid_countries else oda_countries

    # Small states groups
    if getattr(filters, 'small_states_groups', None):
        ss_countries: Set[str] = set()
        for group in filters.small_states_groups:
            if group == 'Small States':
                ss_countries.update(METADATA.get('small_states', set()))
            elif group == 'SSF':
                ss_countries.update(METADATA.get('ssf_members', set()))
            elif group == 'UN SIDS':
                ss_countries.update(METADATA.get('un_sids_members', set()))
        valid_countries = (valid_countries & ss_countries) if valid_countries else ss_countries

    # Huntington civilizations
    if getattr(filters, 'huntington_civilizations', None):
        hunt_countries = {
            code for code, civ in METADATA.get('huntington', {}).items()
            if civ in filters.huntington_civilizations
        }
        valid_countries = (valid_countries & hunt_countries) if valid_countries else hunt_countries

    # Apply country filter if any countries are constrained
    if valid_countries:
        filtered_docs = filtered_docs[filtered_docs['ISO Alpha-3 Code'].isin(valid_countries)]

    # Doc type filter
    if getattr(filters, 'doc_types', None):
        filtered_docs = filtered_docs[filtered_docs['NSS/WP/DD'].isin(filters.doc_types)]

    # Year range filter
    if getattr(filters, 'year_min', None) is not None:
        filtered_docs = filtered_docs[filtered_docs['Year'] >= filters.year_min]
    if getattr(filters, 'year_max', None) is not None:
        filtered_docs = filtered_docs[filtered_docs['Year'] <= filters.year_max]

    # Most recent per country
    if getattr(filters, 'most_recent_only', False):
        filtered_docs = (
            filtered_docs.sort_values('Year')
            .groupby('ISO Alpha-3 Code')
            .last()
            .reset_index()
        )

    return filtered_docs['File prefix'].astype(str).tolist()


def enrich_result_with_metadata(result: 'SearchResult') -> 'SearchResult':
    """Add region/subregion/income/freedom metadata to a SearchResult (in place)."""
    country_code = getattr(result, 'country_iso', '') or ''
    result.region = METADATA.get('regions', {}).get(country_code, '')
    result.subregion = METADATA.get('subregions', {}).get(country_code, '')
    result.income_group = METADATA.get('income_groups', {}).get(country_code, '')
    result.freedom_status = METADATA.get('freedom_house', {}).get(country_code, '')
    return result


def get_segment_text_safe(segment_data: Dict) -> Optional[str]:
    """Safely extract text from segment dict with fallback keys."""
    if not segment_data:
        return None
    for key in ['cleaned', 'text', 'segment', 'content', 'sentence', 'raw']:
        val = segment_data.get(key)
        if val:
            return str(val).strip()
    return None


def create_keyword_pattern(keyword: str, smart_boundaries: bool = True) -> re.Pattern:
    """
    Create a compiled regex pattern for keyword/phrase matching with wildcard support.

    - Robust tokenisation: split ORIGINAL keyword on whitespace, then escape tokens.
    - Wildcards:
        * -> matches any characters (non-greedy)
        ? -> matches any single character
    - Smart phrase matching (optional): allows unicode dashes and separators between tokens:
      -, /, – (U+2013), — (U+2014), ‑ (U+2011), − (U+2212), soft hyphen (U+00AD), plus whitespace.
    - Safer boundaries: uses letter/digit boundaries rather than \\b.

    Returns:
        Compiled pattern (IGNORECASE | MULTILINE).
    """
    MATCH_NOTHING = re.compile(r'(?!)')

    if keyword is None:
        return MATCH_NOTHING

    keyword = str(keyword).strip()
    if not keyword:
        return MATCH_NOTHING

    raw_tokens = re.findall(r"\S+", keyword)
    if not raw_tokens:
        return MATCH_NOTHING

    def token_to_regex(token: str) -> str:
        escaped = re.escape(token)
        escaped = escaped.replace(r"\*", r".*?")  # non-greedy wildcard
        escaped = escaped.replace(r"\?", r".")    # single char wildcard
        return escaped

    tokens = [token_to_regex(t) for t in raw_tokens]

    sep = r"[\s\-/–—‑−\u00AD]*"
    start_boundary = r"(?<![A-Za-z0-9])"
    end_boundary = r"(?![A-Za-z0-9])"

    if len(tokens) == 1:
        pattern_str = start_boundary + tokens[0] + end_boundary
    else:
        if smart_boundaries:
            body = sep.join(tokens)
        else:
            body = r"\s+".join(tokens)
        pattern_str = start_boundary + body + end_boundary

    return re.compile(pattern_str, re.IGNORECASE | re.MULTILINE)


def get_context_window(segment_id: str, model_dict: Dict, window_size: int = 1) -> Tuple[str, str]:
    """
    Get surrounding segment text (before, after) for a segment_id like "DOCID/12".
    """
    if window_size <= 0:
        return '', ''

    if not model_dict or 'segments_dict' not in model_dict:
        return '', ''

    try:
        doc_id, segment_num = segment_id.split('/')
        segment_num = int(segment_num)
    except Exception:
        return '', ''

    segs = model_dict['segments_dict']
    before_parts = []
    after_parts = []

    for i in range(max(1, segment_num - window_size), segment_num):
        sid = f"{doc_id}/{i}"
        txt = get_segment_text_safe(segs.get(sid, {}))
        if txt:
            before_parts.append(txt)

    for i in range(segment_num + 1, segment_num + window_size + 1):
        sid = f"{doc_id}/{i}"
        txt = get_segment_text_safe(segs.get(sid, {}))
        if txt:
            after_parts.append(txt)

    return ' '.join(before_parts), ' '.join(after_parts)


print('✅ Section 4 loaded: initialize_model, apply_metadata_filters, keyword helpers, context helpers')

In [ ]:
%%capture
def extract_search_terms_for_stopwords(query: str) -> set:
    """
    Extract meaningful terms from search query to exclude from cluster names.
    
    Expands terms to include common word forms.
    
    Args:
        query: Search query string
    
    Returns:
        Set of terms to exclude from cluster names
    """
    import re
    
    # Query-level stopwords (not meaningful)
    query_stopwords = {'is', 'are', 'was', 'were', 'a', 'an', 'the', 'as', 'to',
                      'for', 'in', 'on', 'at', 'of', 'and', 'or', 'but', 'new'}
    
    # Tokenise - get words 4+ chars
    tokens = re.findall(r'\b[a-zA-Z]{4,}\b', query.lower())
    
    # Expand to include word variations
    meaningful_terms = set()
    for token in tokens:
        if token not in query_stopwords:
            meaningful_terms.add(token)
            # Add common variations to catch related forms
            if token.endswith('ism'):
                meaningful_terms.add(token[:-3])  # terrorism -> terror
            if token.endswith('ist'):
                meaningful_terms.add(token[:-3])  # extremist -> extrem
            if token.endswith('ists'):
                meaningful_terms.add(token[:-4]) # terrorists -> terror
    
    return meaningful_terms

def deduplicate_cluster_name(name: str) -> str:
    """Remove consecutive duplicate words from cluster name."""
    words = name.split()
    if not words:
        return name
    
    deduped = [words[0]]
    for word in words[1:]:
        if word.lower() != deduped[-1].lower():
            deduped.append(word)
    
    return ' '.join(deduped)

def should_filter_term(term: str, stopwords: set) -> bool:
    """
    Check if a term should be filtered based on stopwords.
    
    For single words: exact match
    For bigrams: filter only if ALL words are stopwords
    """
    words = term.lower().split()
    
    if len(words) == 1:
        # Single word: exact match against stopwords
        return words[0] in stopwords
    else:
        # Bigram/multi-word: filter only if ALL words are stopwords
        return all(word in stopwords for word in words)

def generate_cluster_names_tfidf(results: List[SearchResult], cluster_info: Dict,
                                 search_query: str = '',  # NEW parameter
                                 max_terms: int = 3, min_word_length: int = 4) -> Dict[str, str]:
    """
    Generate descriptive cluster names using TF-IDF analysis.
    
    Args:
        results: List of SearchResult objects with cluster_id assigned
        cluster_info: Dictionary containing cluster_dict and cluster_labels
        search_query: Search query to exclude from cluster names (NEW)
        max_terms: Maximum number of terms to use in cluster name (default: 3)
        min_word_length: Minimum character length for words to consider (default: 4)
    
    Returns:
        Dictionary mapping cluster_id -> cluster_name
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from collections import defaultdict
    import re
    
    if not results or not cluster_info:
        return {}
    
    # Group results by cluster
    cluster_texts = defaultdict(list)
    for result in results:
        cluster_id = result.cluster_id
        if cluster_id is not None:
            cluster_texts[cluster_id].append(result.text)
    
    # If no clusters (all singletons), return empty mapping
    if not cluster_texts or (len(cluster_texts) == 1 and 'singletons' in cluster_texts):
        return {'singletons': 'Unclustered'}
    
    # Custom stopwords for security domain (UK spelling) - BALANCED SET
    security_stopwords = {
        'threat', 'threats', 'security', 'national', 'defence', 'defense',
        'risk', 'risks', 'challenge', 'challenges', 'issue', 'issues',
        'will', 'must', 'need', 'make', 'ensure', 'continue', 'work',
        'including', 'across', 'through', 'within', 'against', 'would',
        'could', 'should', 'also', 'these', 'those', 'their', 'this',
        'that', 'with', 'from', 'have', 'been', 'more', 'such', 'other',
        'into', 'over', 'than', 'some', 'what', 'which', 'when', 'where',
        'while', 'about', 'between', 'during', 'before', 'after',
        'year', 'years', 'time', 'country', 'countries', 'state', 'states',
        'government', 'governments', 'nation', 'nations', 'people',
        # NEW: Only the most generic terms (reduced set)
        'form', 'forms', 'type', 'types',
        'activity', 'activities',
        'group', 'groups',
        'increase', 'increased', 'remain', 'remains', 'pose', 'posed'
    }
    
    # NEW: Add search query terms to stopwords
    if search_query:
        query_terms = extract_search_terms_for_stopwords(search_query)
        security_stopwords.update(query_terms)
    
    # Combine cluster texts into documents
    cluster_ids = []
    cluster_documents = []
    
    for cluster_id, texts in cluster_texts.items():
        if cluster_id == 'singletons':
            continue
        cluster_ids.append(cluster_id)
        # Concatenate all texts in cluster
        cluster_documents.append(' '.join(texts))
    
    if len(cluster_documents) < 2:
        # If only one cluster, generate name from most frequent words
        if len(cluster_documents) == 1:
            return {cluster_ids[0]: _extract_top_words_simple(cluster_documents[0], 
                                                               max_terms, 
                                                               min_word_length, 
                                                               security_stopwords)}
        return {}
    
    # Create TF-IDF vectoriser
    vectoriser = TfidfVectorizer(
        max_features=100,
        stop_words='english',
        ngram_range=(1, 2),  # Include bigrams
        min_df=1,
        max_df=0.8,
        lowercase=True,
        token_pattern=r'\b[a-zA-Z]{' + str(min_word_length) + r',}\b'  # Min word length
    )
    
    try:
        # Fit and transform
        tfidf_matrix = vectoriser.fit_transform(cluster_documents)
        feature_names = vectoriser.get_feature_names_out()
        
        # Generate names for each cluster
        cluster_names = {}
        
        for idx, cluster_id in enumerate(cluster_ids):
            # Get TF-IDF scores for this cluster
            tfidf_scores = tfidf_matrix[idx].toarray()[0]
            
            # Get top terms (increased candidate pool)
            top_indices = tfidf_scores.argsort()[-max_terms*5:][::-1]  # Get 5x candidates instead of 3x
            
            # Filter using improved logic
            top_terms = []
            for i in top_indices:
                term = feature_names[i]
                # Use improved filtering that distinguishes single words from bigrams
                if should_filter_term(term, security_stopwords):
                    continue
                top_terms.append(term)
                if len(top_terms) >= max_terms:
                    break
            
            # Capitalise and join
            cluster_name = ' '.join(word.capitalize() for word in top_terms)
            
            # NEW: Remove duplicate words
            cluster_name = deduplicate_cluster_name(cluster_name)
            
            # Fallback if no terms found
            if not cluster_name:
                cluster_name = f'Cluster {cluster_id}'
            
            cluster_names[cluster_id] = cluster_name
        
        # Add singletons label
        if 'singletons' in cluster_texts:
            cluster_names['singletons'] = 'Unclustered'
        
        return cluster_names
        
    except Exception as e:
        print(f'⚠️  Error generating cluster names: {e}')
        # Fallback to simple naming
        return {cluster_id: f'Cluster {cluster_id}' for cluster_id in cluster_ids}

def _extract_top_words_simple(text: str, max_terms: int, min_word_length: int, 
                               stopwords: set) -> str:
    """
    Simple word frequency extraction fallback.
    
    Args:
        text: Text to analyse
        max_terms: Number of terms to extract
        min_word_length: Minimum word length
        stopwords: Words to exclude
    
    Returns:
        Formatted cluster name
    """
    from collections import Counter
    import re
    
    # Tokenise and clean
    words = re.findall(r'\b[a-zA-Z]{' + str(min_word_length) + r',}\b', text.lower())
    
    # Filter stopwords (exact match)
    words = [w for w in words if w not in stopwords]
    
    # Count frequencies
    word_counts = Counter(words)
    
    # Get top words
    top_words = [word for word, count in word_counts.most_common(max_terms)]
    
    # Capitalise and format
    return ' '.join(word.capitalize() for word in top_words) if top_words else 'Cluster'

def apply_cluster_names(results: List[SearchResult], cluster_names: Dict[str, str]):
    """
    Apply cluster names to SearchResult objects in place.
    
    Args:
        results: List of SearchResult objects
        cluster_names: Dictionary mapping cluster_id -> cluster_name
    """
    for result in results:
        cluster_id = result.cluster_id
        if cluster_id is not None and cluster_id in cluster_names:
            result.cluster_name = cluster_names[cluster_id]
        elif cluster_id == 'singletons':
            result.cluster_name = cluster_names.get('singletons', 'Unclustered')
        else:
            result.cluster_name = f'Cluster {cluster_id}' if cluster_id != 'singletons' else 'Unclustered'

print('✅ TF-IDF cluster naming functions defined')

def cluster_search_results_internal(
    results: List[SearchResult],
    model_dict: Dict,
    cluster_threshold: float = 0.78,
    search_query: str = ''
) -> Dict:
    """
    Shared clustering function for both similarity and keyword searches.

    Extracts MPNet embeddings for result segments, computes pairwise similarity,
    applies threshold to create clusters, and generates descriptive cluster names.

    Args:
        results: List of SearchResult objects with segment_id populated
        model_dict: Dictionary containing segment_encodings and encoded_segments
        cluster_threshold: Similarity threshold for clustering (default 0.78)
        search_query: Query terms to exclude from cluster names

    Returns:
        Dictionary with keys:
            - cluster_dict: {cluster_id: [result_indices]} mapping
            - cluster_labels: array of cluster_id per result
            - num_clusters: count of clusters with 2+ members
            - num_singletons: count of unclustered results
            - cluster_names: {cluster_id: descriptive_name} mapping
    """
    from scipy.spatial.distance import pdist, squareform
    from scipy.sparse.csgraph import connected_components
    import numpy as np

    # Extract embeddings for result segments
    segment_encodings = np.array(model_dict['segment_encodings'])
    encoded_segments = model_dict['encoded_segments']

    # Map segment IDs to encoding indices
    segment_to_idx = {seg_id: idx for idx, seg_id in enumerate(encoded_segments)}

    result_embeddings = []
    valid_indices = []

    for i, result in enumerate(results):
        seg_id = result.segment_id
        if seg_id in segment_to_idx:
            enc_idx = segment_to_idx[seg_id]
            result_embeddings.append(segment_encodings[enc_idx])
            valid_indices.append(i)

    if len(result_embeddings) == 0:
        raise ValueError('No valid embeddings found for results')

    result_embeddings = np.array(result_embeddings)

    # Compute pairwise cosine distance
    distances = pdist(result_embeddings, metric='cosine')

    # Convert to angular distance: 1 - (arccos(1 - cosine_dist) / pi)
    angular_distances = 1.0 - (np.arccos(np.clip(1.0 - distances, -1.0, 1.0)) / np.pi)

    # Convert to similarity matrix
    similarity_matrix = squareform(angular_distances)

    # Apply threshold to create adjacency matrix
    adjacency = (similarity_matrix >= cluster_threshold).astype(int)
    np.fill_diagonal(adjacency, 0)  # Remove self-connections

    # Find connected components
    n_components, labels = connected_components(
        csgraph=adjacency,
        directed=False,
        return_labels=True
    )

    # Map back to original result indices
    full_labels = np.full(len(results), -1, dtype=int)
    for i, valid_idx in enumerate(valid_indices):
        full_labels[valid_idx] = labels[i]

    # Separate clusters (2+ members) from singletons
    cluster_dict = {}
    singleton_indices = []

    # Create mapping from label to original result indices
    label_to_indices = {}
    for i, valid_idx in enumerate(valid_indices):
        label = labels[i]
        if label not in label_to_indices:
            label_to_indices[label] = []
        label_to_indices[label].append(valid_idx)

    # Separate clusters (2+ members) from singletons
    for label, indices in label_to_indices.items():
        if len(indices) >= 2:
            cluster_id = f'cluster_{label}'
            cluster_dict[cluster_id] = indices
        else:
            singleton_indices.extend(indices)

    # Add any results without valid embeddings to singletons
    for i in range(len(results)):
        if i not in valid_indices:
            singleton_indices.append(i)

    if singleton_indices:
        cluster_dict['singletons'] = sorted(set(singleton_indices))

    # Build cluster info dictionary
    cluster_info = {
        'cluster_dict': cluster_dict,
        'cluster_labels': full_labels,
        'num_clusters': len([k for k in cluster_dict.keys() if k != 'singletons']),
        'num_singletons': len(singleton_indices)
    }

    # Assign cluster IDs to results FIRST (needed for name generation)
    for cluster_id, indices in cluster_dict.items():
        for idx in indices:
            results[idx].cluster_id = cluster_id if cluster_id != 'singletons' else 'singletons'

    # Generate cluster names using TF-IDF (requires cluster_id on results)
    cluster_names = generate_cluster_names_tfidf(results, cluster_info, search_query)
    cluster_info['cluster_names'] = cluster_names

    # Apply cluster names to results
    apply_cluster_names(results, cluster_names)

    return cluster_info


In [ ]:
%%capture
def display_filter_summary(filters: MetadataFilters):
    """Display active filters in an attractive format."""
    summary = filters.get_summary()
    
    if summary == 'No filters applied':
        style = 'background-color: #f0f0f0; padding: 10px; border-radius: 5px; color: #666;'
        icon = '🌐'
    else:
        style = 'background-color: #e3f2fd; padding: 10px; border-radius: 5px; border-left: 4px solid #2196F3; color: #1565c0;'
        icon = '🔍'
    
    html = f"""
    <div style="{style}">
        <strong>{icon} Active Filters:</strong><br/>
        {summary}
    </div>
    """
    display(HTML(html))

def display_search_results(results: List[SearchResult], topic: str, filters: MetadataFilters, 
                          config: SearchConfig):
    """Display search results with metadata and filter information."""
    
    # Header
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                padding: 20px; border-radius: 10px; color: white; margin: 20px 0;">
        <h2 style="margin: 0; font-size: 24px;">🔍 Results for: "{topic}"</h2>
        <p style="margin: 10px 0 0 0; opacity: 0.9;">Found {len(results)} results in {len(set(r.document_id for r in results))} documents</p>
    </div>
    """))
    
    # Display active filters
    display_filter_summary(filters)
    
    # Search configuration
    search_type = 'Keyword Search' if config.keyword_search else 'Semantic Search'
    display(HTML(f"""
    <div style="background-color: #fff3e0; padding: 10px; margin: 10px 0; 
                border-radius: 5px; border-left: 4px solid #ff9800;">
        <strong>⚙️ Search Configuration:</strong> {search_type} | 
        Threshold: {config.search_threshold:.2f} | 
        Cluster: {config.cluster_threshold:.2f}
    </div>
    """))
    
    # Group results by document
    doc_groups = {}
    for r in results:
        if r.document_id not in doc_groups:
            doc_groups[r.document_id] = []
        doc_groups[r.document_id].append(r)
    
    # Display each document group
    for doc_id, doc_results in sorted(doc_groups.items()):
        first_result = doc_results[0]
        
        # Document header with metadata
        metadata_tags = []
        if first_result.region:
            metadata_tags.append(f"🌍 {first_result.region}")
        if first_result.subregion:
            metadata_tags.append(f"📍 {first_result.subregion}")
        if first_result.income_group:
            metadata_tags.append(f"💰 {first_result.income_group}")
        if first_result.freedom_status:
            metadata_tags.append(f"🗳️ {first_result.freedom_status}")
        
        metadata_str = ' | '.join(metadata_tags)
        
        display(HTML(f"""
        <div style="background-color: #f5f5f5; padding: 15px; margin: 15px 0; 
                    border-radius: 8px; border-left: 5px solid #4CAF50;">
            <h3 style="margin: 0 0 10px 0; color: #546e7a;">
                📄 {first_result.country} ({first_result.year})
            </h3>
            <p style="margin: 5px 0; color: #555; font-size: 14px;">
                <strong>{first_result.document_name}</strong>
            </p>
            <p style="margin: 5px 0; color: #777; font-size: 12px;">
                {metadata_str}
            </p>
            <p style="margin: 5px 0; color: #999; font-size: 12px;">
                📊 {len(doc_results)} results in this document
            </p>
        </div>
        """))
        
        # Display individual results
        for i, result in enumerate(doc_results[:5], 1):  # Show top 5 per document
            cluster_info = f" | Cluster: {result.cluster_id}" if result.cluster_id is not None else ""
            display(HTML(f"""
            <div style="background-color: white; padding: 12px; margin: 8px 0 8px 20px; 
                        border-radius: 5px; border-left: 3px solid #9C27B0;">
                <p style="margin: 0 0 5px 0; color: #888; font-size: 11px;">
                    Result {i} | Score: {result.score:.3f}{cluster_info}
                </p>
                <p style="margin: 0; color: #333; line-height: 1.6;">
                    {result.text[:500]}{'...' if len(result.text) > 500 else ''}
                </p>
            </div>
            """))
        
        if len(doc_results) > 5:
            display(HTML(f"""
            <p style="margin: 5px 0 5px 20px; color: #999; font-size: 12px; font-style: italic;">
                ... and {len(doc_results) - 5} more results
            </p>
            """))

def create_results_summary_chart(all_results: Dict[str, List[SearchResult]], filters: MetadataFilters):
    """Create summary visualization of search results."""
    if not all_results:
        return
    
    # Aggregate statistics
    topic_counts = {topic: len(results) for topic, results in all_results.items()}
    
    # Create bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    topics = list(topic_counts.keys())
    counts = list(topic_counts.values())
    
    bars = ax.barh(topics, counts, color='#667eea')
    ax.set_xlabel('Number of Results', fontsize=12)
    ax.set_title('Search Results Summary', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, 
               f' {int(width)}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Display filter summary
    display(Markdown(f"**Filters applied:** {filters.get_summary()}"))

print('✅ Display functions defined')

In [ ]:
%%capture
def generate_search_visualisations(all_results, all_cluster_info, save_dir=None, file_prefix=None):
    """Generate SVG charts from search results.

    Returns an HTML string with the charts embedded inline.
    If *save_dir* is provided the three SVG files are also written to disk
    alongside the exported CSVs.
    """
    import io
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from collections import Counter
    from pathlib import Path

    # Flatten all results across topics
    flat_results = [r for results in all_results.values() for r in results]
    if not flat_results:
        return ''

    plt.rcParams.update({
        'font.family': 'sans-serif',
        'axes.spines.top': False,
        'axes.spines.right': False,
    })
    COLOUR = '#667eea'
    svg_parts = []
    saved_paths = []

    prefix = file_prefix or 'search'

    def fig_to_svg(fig, filename=None):
        if save_dir and filename:
            out_path = Path(save_dir) / filename
            out_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(out_path, format='svg', bbox_inches='tight')
            saved_paths.append(str(out_path))
        buf = io.StringIO()
        fig.savefig(buf, format='svg', bbox_inches='tight')
        plt.close(fig)
        return buf.getvalue()

    # ── Chart 1: Results by year ──────────────────────────────────────────
    year_counts = Counter(r.year for r in flat_results if r.year)
    if year_counts:
        years = sorted(year_counts)
        counts = [year_counts[y] for y in years]
        fig, ax = plt.subplots(figsize=(10, 3.5))
        ax.bar(years, counts, color=COLOUR, width=0.7)
        ax.set_xlabel('Publication year')
        ax.set_ylabel('Results')
        ax.set_title('Results by year')
        ax.set_xticks(years)
        ax.tick_params(axis='x', rotation=45)
        fig.tight_layout()
        svg_parts.append(fig_to_svg(fig, f'{prefix}_by_year.svg'))
        svg_parts.append(
            '<p style="font-size:0.82em;color:#666;margin:4px 0 16px 0;">'
            '<strong>Note on the chart above:</strong> The apparent upward trend partly reflects '
            'growth in the number of documents published each year across the dataset as a whole '
            '&mdash; not necessarily increased salience of the search topic. Results are also '
            'shaped by uneven country coverage: some countries have up to 31 documents spanning '
            '38 years, while 35 countries have only one document (median: 3 per country). '
            'Filtering by country or region can help control for this.</p>'
        )

    # ── Chart 2: Top countries ────────────────────────────────────────────
    country_counts = Counter(r.country for r in flat_results if r.country)
    if country_counts:
        top = country_counts.most_common(15)
        labels = [c for c, _ in reversed(top)]
        values = [v for _, v in reversed(top)]
        fig, ax = plt.subplots(figsize=(8, max(3, len(labels) * 0.35)))
        ax.barh(labels, values, color=COLOUR)
        ax.set_xlabel('Results')
        ax.set_title('Top countries')
        fig.tight_layout()
        svg_parts.append(fig_to_svg(fig, f'{prefix}_top_countries.svg'))

    # ── Chart 3: Cluster overview ─────────────────────────────────────────
    cluster_sizes = []
    for topic, ci in all_cluster_info.items():
        cd = ci.get('cluster_dict', {})
        names = ci.get('cluster_names', {})
        for cid, results in cd.items():
            label = names.get(cid, str(cid))
            cluster_sizes.append((label, len(results)))
    if cluster_sizes:
        cluster_sizes.sort(key=lambda x: x[1])
        labels = [l for l, _ in cluster_sizes]
        values = [v for _, v in cluster_sizes]
        fig, ax = plt.subplots(figsize=(8, max(3, len(labels) * 0.32)))
        ax.barh(labels, values, color=COLOUR)
        ax.set_xlabel('Results')
        ax.set_title('Cluster overview')
        fig.tight_layout()
        svg_parts.append(fig_to_svg(fig, f'{prefix}_cluster_overview.svg'))

    if not svg_parts:
        return ''

    if saved_paths:
        print(f'  ✓ Saved {len(saved_paths)} SVG charts to {save_dir}/')

    html = '<div style="margin-top: 30px;">'
    html += '<h3 style="color: #546e7a; font-family: sans-serif;">📊 Visualisations</h3>'
    for svg in svg_parts:
        html += f'<div style="margin-bottom: 20px;">{svg}</div>'
    html += '</div>'
    return html

print('✅ Visualisation function loaded')

In [ ]:
%%capture

def create_search_interface():
    """Create the main search interface with all filtering options."""

    # Title
    title = widgets.HTML(
        value="""
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    padding: 25px; border-radius: 10px; color: white; text-align: center;">
            <h1 style="margin: 0;">🌍 Metadata Search Interface</h1>
            <p style="margin: 10px 0 0 0; opacity: 0.9;">Search with comprehensive filtering options</p>
        </div>
        """
    )

    # Search configuration section
    search_header = widgets.HTML(
        value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>🔍 Search Configuration</h3>"
    )

    topics_widget = widgets.Textarea(
        value='',
        placeholder='Enter search topics (one per line)\nExample:\ncyber threats\nclimate security',
        description='Topics:',
        layout=Layout(width='80%', height='100px')
    )

    search_threshold = widgets.FloatSlider(
        value=0.70,
        min=0.50,
        max=0.90,
        step=0.01,
        description='Search threshold:',
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )

    cluster_threshold = widgets.FloatSlider(
        value=0.78,
        min=0.50,
        max=0.95,
        step=0.005,
        description='Cluster threshold (applies to both search modes):',
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )

    semantic_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #fff9e6; border-left: 3px solid #ffa726;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Semantic Search Tips:</b><br>
            • <b>How it works:</b> Uses AI to understand meaning, not just keywords<br>
            • <b>Best with:</b> Complete sentences or phrases (e.g., 'climate change poses security threats')<br>
            • <b>Why sentences:</b> Provides context that improves semantic matching accuracy<br>
            • <b>Search threshold:</b> 0.70 = broad matches, 0.85 = strict/precise matches<br>
            • <b>Example:</b> 'cyber attacks' finds 'digital threats', 'online security breaches', etc.<br>
            • <b>Use when:</b> Exploring concepts, finding related ideas, or researching topics
        </div>
    """)

    clustering_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #e8f5e9; border-left: 3px solid #4caf50;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Clustering (Both Search Modes):</b><br>
            • <b>What it does:</b> Groups similar results together with descriptive labels<br>
            • <b>Applies to:</b> Both keyword and semantic search results<br>
            • <b>Cluster threshold:</b> 0.78 = balanced, 0.85 = strict grouping, 0.70 = loose grouping<br>
            • <b>How it works:</b> Uses semantic similarity (MPNet embeddings) to find related segments<br>
            • <b>Cluster naming:</b> Automatically generates descriptive labels using TF-IDF analysis<br>
            • <b>Name generation:</b> Extracts the most distinctive terms from each cluster's text<br>
            • <b>Example names:</b> "Development Aid Security", "Climate Environmental Change", "Military Operations"<br>
            • <b>Benefit:</b> Organizes large result sets into meaningful themes with readable labels
        </div>
    """)

    # Search mode section
    mode_header = widgets.HTML(
        value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>🔎 Search Mode</h3>"
    )

    keyword_search_widget = widgets.Checkbox(
        value=False,
        description='Use keyword search (regex) instead of semantic search',
        indent=False,
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    keyword_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #f0f4f8; border-left: 3px solid #4c6ef5;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Keyword Search Tips:</b><br>
            • <b>Wildcards:</b> Use <code>*</code> for any characters, <code>?</code> for single character<br>
            • <b>Phrases/N-grams:</b> Multi-word phrases work (e.g., <code>climate change</code>)<br>
            • <b>Case-insensitive:</b> Search ignores case by default<br>
            • <b>Word boundaries:</b> Matches whole words only<br>
            • <b>Special characters:</b> Most are automatically escaped
        </div>
    """)

    boolean_mode_widget = widgets.Checkbox(
        value=False,
        description='Enable Boolean operators (AND, OR, NOT)',
        indent=False,
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    boolean_scoring_widget = widgets.Dropdown(
        options=[('Match count (default)', 'match_count'),
                 ('Average score', 'avg_score'),
                 ('Maximum score', 'max_score')],
        value='match_count',
        description='Boolean scoring:',
        style={'description_width': 'initial'},
        layout=Layout(width='400px')
    )

    boolean_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #f0f4f8; border-left: 3px solid #7c3aed;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Boolean Search Operators:</b><br>
            • <b>AND</b> (<code>&</code>), <b>OR</b> (<code>|</code>), <b>NOT</b> (<code>!</code>)<br>
            • <b>Parentheses</b>: Control precedence — <code>(cyber OR digital) AND threat</code><br>
            • <b>Quotes</b>: Exact phrases — <code>"climate change" AND security</code>
        </div>
    """)

    fuzzy_matching_widget = widgets.Checkbox(
        value=False,
        description='Enable fuzzy matching (for keyword search fallback)',
        indent=False,
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    smart_boundaries_widget = widgets.Checkbox(
        value=True,
        description='Smart phrase matching (handles hyphens/punctuation)',
        indent=False,
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    fuzzy_threshold_widget = widgets.IntSlider(
        value=80,
        min=50,
        max=100,
        step=5,
        description='Fuzzy threshold:',
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )

    fuzzy_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #f0f4f8; border-left: 3px solid #4c6ef5;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Fuzzy Matching Tips:</b> Matches approximate terms (catches typos). 80% is recommended.
        </div>
    """)

    smart_boundaries_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #f0f4f8; border-left: 3px solid #4c6ef5;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Smart Phrase Matching:</b> Handles variation like "cyber-security" vs "cyber security".
        </div>
    """)

    context_window_widget = widgets.IntSlider(
        value=1,
        min=0,
        max=5,
        step=1,
        description='Context window:',
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )

    geo_header = widgets.HTML(value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>🌐 Geographic Filters</h3>")

    regions_widget = widgets.SelectMultiple(
        options=METADATA.get('unique_regions', []),
        description='Regions:',
        layout=Layout(width='60%', height='100px')
    )
    subregions_widget = widgets.SelectMultiple(
        options=METADATA.get('unique_subregions', []),
        description='Subregions:',
        layout=Layout(width='60%', height='120px')
    )

    # Initialize country lookup
    country_lookup = CountryLookup()
    all_countries = country_lookup.list_all_countries(show_region=False)
    country_options = [(f"{num:3d}. {iso:3s} {name}", iso) for num, iso, name, _ in all_countries]

    countries_widget = widgets.SelectMultiple(
        options=country_options,
        description='Countries:',
        layout=Layout(width='80%', height='200px'),
        style={'description_width': 'initial'}
    )

    countries_help = widgets.HTML(value="""
        <div style="margin: 5px 0 15px 20px; padding: 8px 12px;
                    background-color: #e8f5e9; border-left: 3px solid #66bb6a;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Country Selection:</b><br>
            • Select one or more countries from the list (1-118)<br>
            • Click to select, Ctrl/Cmd+Click for multiple, Shift+Click for ranges<br>
            • Leave empty to search all countries<br>
            • Combine with region/subregion filters for more control
        </div>
    """)

    org_header = widgets.HTML(value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>🤝 International Organizations</h3>")
    organizations_widget = widgets.SelectMultiple(
        options=sorted(METADATA.get('organizations', {}).keys()),
        description='Organizations:',
        layout=Layout(width='60%', height='150px')
    )

    econ_header = widgets.HTML(
        value="<h3 style='margin: 20px 0 10px 0; color: #546e7a; font-weight: bold;'>💰 Economic & Political Filters</h3>"
    )

    time_based_note = widgets.HTML(
        value="""
        <div style='margin: 10px 0; padding: 12px;
                    background-color: #e3f2fd;
                    border-left: 5px solid #1565c0;
                    color: #0d47a1;
                    font-size: 14px;'>
            <strong>Note:</strong> Uses latest available status.
        </div>
        """
    )
    income_widget = widgets.SelectMultiple(options=['High', 'Upper-middle', 'Lower-middle', 'Low'], description='Income groups:', layout=Layout(width='60%', height='100px'))
    freedom_widget = widgets.SelectMultiple(options=['Free', 'Partly Free', 'Not Free'], description='Democracy:', layout=Layout(width='60%', height='80px'))
    oda_widget = widgets.SelectMultiple(options=['ODA_Recipient', 'Non_Recipient'], description='ODA status:', layout=Layout(width='60%', height='80px'))
    small_states_widget = widgets.SelectMultiple(options=['Small States', 'SSF', 'UN SIDS'], description='Small States:', layout=Layout(width='60%', height='80px'))
    huntington_widget = widgets.SelectMultiple(options=METADATA.get('huntington_civilizations', []), description='Huntington', layout=Layout(width='60%', height='120px'))

    doc_header = widgets.HTML(value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>📄 Document Filters</h3>")
    doc_types_widget = widgets.SelectMultiple(options=['NSS', 'WP', 'DD', 'TA'], description='Doc types:', layout=Layout(width='60%', height='100px'))
    year_range_widget = widgets.IntRangeSlider(value=[1987, 2025], min=1987, max=2025, step=1, description='Year range:', layout=Layout(width='60%'))
    most_recent_widget = widgets.Checkbox(value=False, description='Most recent document per country only', layout=Layout(width='max-content'))

    # CSV Export widgets
    export_csv_widget = widgets.Checkbox(
        value=True,
        description='Export results to CSV',
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    iterative_export_widget = widgets.Checkbox(
        value=False,
        description='Use iterative filenames (add timestamp to avoid overwriting)',
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    export_warning = widgets.HTML(
        value='<div style="color: #d32f2f; font-size: 0.9em; margin-top: 5px;">⚠️ Searches with the same name will overwrite previous CSV files unless iterative filenames are enabled.</div>',
        layout=Layout(width='100%', margin='5px 0')
    )

    search_button = Button(description='🔍 Search', button_style='success', layout=Layout(width='200px', height='50px'), style={'button_color': '#4CAF50', 'font_weight': 'bold'})
    output = widgets.Output()
    charts_widget = widgets.HTML(value='')

    def on_search_click(b):
        charts_widget.value = ''
        with output:
            output.clear_output()
            if MODEL_DICT is None:
                display(HTML('<div style="color: #c62828;">⚠️ Model not loaded.</div>'))
                return

            topics = [t.strip() for t in topics_widget.value.split('\n') if t.strip()]
            if not topics:
                display(HTML('<div style="color: #c62828;">⚠️ Please enter a topic.</div>'))
                return

            filters = MetadataFilters(
                regions=list(regions_widget.value),
                subregions=list(subregions_widget.value),
                countries=list(countries_widget.value),
                organizations=list(organizations_widget.value),
                income_groups=list(income_widget.value),
                freedom_house_status=list(freedom_widget.value),
                oda_status=list(oda_widget.value),
                small_states_groups=list(small_states_widget.value),
                huntington_civilizations=list(huntington_widget.value),
                doc_types=list(doc_types_widget.value),
                year_min=year_range_widget.value[0],
                year_max=year_range_widget.value[1],
                most_recent_only=most_recent_widget.value
            )

            config = SearchConfig(
                topics=topics,
                search_threshold=search_threshold.value,
                cluster_threshold=cluster_threshold.value,
                filters=filters,
                keyword_search=keyword_search_widget.value,
                boolean_mode=boolean_mode_widget.value,
                boolean_scoring=boolean_scoring_widget.value,
                fuzzy_matching=fuzzy_matching_widget.value,
                fuzzy_threshold=fuzzy_threshold_widget.value,
                smart_boundaries=smart_boundaries_widget.value,
                context_window=context_window_widget.value
            )

            try:
                all_results, all_cluster_info = run_metadata_batch_search(config, MODEL_DICT, ENCODER)
                display_batch_results(all_results, all_cluster_info, config, max_per_topic=5)
                # Derive a safe file prefix from the first topic
                _first_topic = next(iter(all_results), 'search')
                _file_prefix = re.sub(r'_+', '_', re.sub(r'[^\w ]', '_', _first_topic)).strip().replace(' ', '_')[:50].strip('_') or 'search'
                _save_dir = 'outputs' if export_csv_widget.value else None
                charts_widget.value = generate_search_visualisations(all_results, all_cluster_info, save_dir=_save_dir, file_prefix=_file_prefix)
                export_search_results(all_results, all_cluster_info, config,
                                    enable_export=export_csv_widget.value,
                                    iterative_names=iterative_export_widget.value)
            except Exception as e:
                import traceback
                print(f"Error: {e}")
                traceback.print_exc()

    search_button.on_click(on_search_click)

    interface = VBox([
        title, search_header, topics_widget, search_threshold, cluster_threshold,
        search_button,
        semantic_help_widget, clustering_help_widget, mode_header,
        keyword_search_widget, keyword_help_widget, fuzzy_matching_widget,
        fuzzy_threshold_widget, fuzzy_help_widget, smart_boundaries_widget,
        boolean_mode_widget, boolean_scoring_widget, boolean_help_widget,
        smart_boundaries_help_widget, context_window_widget,
        geo_header,
        regions_widget, subregions_widget, countries_widget, countries_help, org_header, organizations_widget,
        econ_header, time_based_note, income_widget, freedom_widget,
        oda_widget, small_states_widget, huntington_widget, doc_header,
        doc_types_widget, year_range_widget, most_recent_widget,
        export_csv_widget,
        iterative_export_widget,
        export_warning,
        output,
        charts_widget
    ])

    return interface

print('✅ Interface functions defined')

In [ ]:
%%capture
# Example: Programmatic search
# This cell demonstrates how to run searches programmatically

# Define search configuration
example_filters = MetadataFilters(
    regions=['Asia'],
    organizations=['ASEAN'],
    year_min=2010,
    year_max=2025
)

example_config = SearchConfig(
    topics=['cyber security', 'maritime security'],
    search_threshold=0.70,
    filters=example_filters
)

print('Example configuration:')
print(f'Topics: {example_config.topics}')
print(f'Filters: {example_config.filters.get_summary()}')

# To run the search, you would call your search function here
# results = run_search(example_config, MODEL_DICT, ENCODER)

In [ ]:
%%capture
# Core similarity search with metadata filtering and clustering
def run_similarity_search(topic: str, config: SearchConfig, model_dict, encoder) -> Tuple[List[SearchResult], Dict]:
    """Run similarity search with metadata filtering and clustering."""
    if not encoder:
        print('❌ Encoder not loaded. Please initialize model first.')
        return [], {}

    # Get filtered documents
    doc_df = METADATA['documents']
    all_doc_ids = doc_df['File prefix'].astype(str).tolist()
    filtered_doc_ids = apply_metadata_filters(all_doc_ids, config.filters)

    if not filtered_doc_ids:
        print('⚠️  No documents match the filter criteria')
        return [], {}

    # Filter segments to selected documents
    encoded_segments = model_dict['encoded_segments']
    segment_encodings = model_dict['segment_encodings']
    segments_dict = model_dict['segments_dict']
    documents_dict = model_dict['documents_dict']
    countries_dict = model_dict['countries_dict']

    # Build filtered indices and segments
    filtered_indices = []
    filtered_segment_ids = []
    for idx, seg_id in enumerate(encoded_segments):
        doc_id = seg_id.split('/')[0]
        if doc_id in filtered_doc_ids:
            filtered_indices.append(idx)
            filtered_segment_ids.append(seg_id)

    if not filtered_indices:
        print('⚠️  No segments found in filtered documents')
        return [], {}

    filtered_encodings = segment_encodings[filtered_indices]

    # Encode query
    query_encoding = encoder.encode(topic)
    query_vector = np.array(query_encoding).flatten()

    # Compute similarities using vectorised formula: 1 - (arccos(1 - cosine_dist) / π)
    query_norm = np.linalg.norm(query_vector)
    enc_norms = np.linalg.norm(filtered_encodings, axis=1)
    dot_products = filtered_encodings @ query_vector
    denom = np.clip(enc_norms * query_norm, 1e-10, None)
    cosine_dists = 1.0 - (dot_products / denom)
    similarities = 1.0 - (np.arccos(np.clip(1.0 - cosine_dists, -1.0, 1.0)) / np.pi)

    # Filter by threshold
    valid_mask = similarities >= config.search_threshold
    valid_segments = [filtered_segment_ids[i] for i in range(len(filtered_segment_ids)) if valid_mask[i]]
    valid_similarities = similarities[valid_mask]

    if len(valid_segments) == 0:
        return [], {}

    # Sort by similarity score (descending)
    segment_similarity_pairs = list(zip(valid_segments, valid_similarities))
    segment_similarity_pairs.sort(key=lambda x: x[1], reverse=True)

    # Apply max results limit if set
    if config.max_results_per_topic is not None:
        segment_similarity_pairs = segment_similarity_pairs[:config.max_results_per_topic]

    # Create search results
    results = []
    for segment_id, similarity in segment_similarity_pairs:
        doc_id = segment_id.split('/')[0]

        # Get segment text
        segment_data = segments_dict.get(segment_id, {})
        segment_text = get_segment_text_safe(segment_data)

        if not segment_text:
            continue

        # Get document metadata
        doc_data = documents_dict.get(doc_id, {})
        country_iso = doc_data.get('iso', '')

        # Get country name from countries_dict
        country_info = countries_dict.get(country_iso, {})
        country_name = country_info.get('country', doc_data.get('country', 'Unknown'))

        # Get context if requested
        context_before, context_after = get_context_window(
            segment_id, model_dict, config.context_window
        )

        result = SearchResult(
            segment_id=segment_id,
            text=segment_text,
            score=float(similarity),
            document_id=doc_id,
            document_name=doc_data.get('document', doc_data.get('name', '')),
            country=country_name,
            country_iso=country_iso,
            year=int(doc_data.get('year', 0)) if str(doc_data.get('year', '')).isdigit() else 0,
            doc_type=doc_data.get('type', ''),
            context_before=context_before,
            context_after=context_after
        )

        # Enrich with metadata
        result = enrich_result_with_metadata(result)
        results.append(result)

    # Perform clustering if we have multiple results
    cluster_info_dict = {}
    if len(results) > 1:
        try:
            cluster_info_dict = cluster_search_results_internal(
                results, model_dict, config.cluster_threshold, search_query=topic
            )
        except Exception as e:
            print(f'⚠️  Clustering error: {str(e)}')
            # Fallback: all singletons
            cluster_info_dict = {
                'cluster_dict': {'singletons': list(range(len(results)))},
                'num_clusters': 0,
                'num_singletons': len(results),
                'cluster_names': {'singletons': 'Unclustered'}
            }
            for result in results:
                result.cluster_id = 'singletons'
                result.cluster_name = 'Unclustered'
    elif len(results) == 1:
        # Single result handling
        cluster_info_dict = {
            'cluster_dict': {'singletons': [0]},
            'num_clusters': 0,
            'num_singletons': 1,
            'cluster_names': {'singletons': 'Unclustered'}
        }
        results[0].cluster_id = 'singletons'
        results[0].cluster_name = 'Unclustered'

    return results, cluster_info_dict

# Keyword search function
def run_keyword_search(keyword: str, config: SearchConfig, model_dict) -> Tuple[List[SearchResult], Dict]:
    """Run keyword search with metadata filtering and optional fuzzy matching."""
    print(f'🔍 Searching for keyword: "{keyword}"')

    # Get filtered documents
    doc_df = METADATA['documents']
    all_doc_ids = doc_df['File prefix'].astype(str).tolist()
    filtered_doc_ids = apply_metadata_filters(all_doc_ids, config.filters)

    if not filtered_doc_ids:
        print('⚠️  No documents match the filter criteria')
        return [], {}

    # Create keyword pattern
    pattern = create_keyword_pattern(keyword, smart_boundaries=config.smart_boundaries)
    results = []

    segments_dict = model_dict['segments_dict']
    documents_dict = model_dict['documents_dict']
    countries_dict = model_dict['countries_dict']
    encoded_segments = model_dict['encoded_segments']

    # Filter segments to selected documents
    relevant_segments = [seg_id for seg_id in encoded_segments
                        if seg_id.split('/')[0] in filtered_doc_ids]

    print(f'📊 Searching {len(relevant_segments):,} segments in {len(filtered_doc_ids)} documents')

    for segment_id in tqdm(relevant_segments, desc=f'Searching "{keyword}"', leave=False):
        doc_id = segment_id.split('/')[0]

        if segment_id not in segments_dict:
            continue

        segment_data = segments_dict[segment_id]
        segment_text = get_segment_text_safe(segment_data)

        if not segment_text:
            continue

        # Check for pattern match
        match_found = False
        match_score = 100.0  # Perfect match score

        if pattern.search(segment_text):
            match_found = True
        elif config.fuzzy_matching and FUZZY_AVAILABLE:
            # Fuzzy matching fallback
            fuzzy_score = fuzz.partial_ratio(keyword.lower(), segment_text.lower())
            if fuzzy_score >= config.fuzzy_threshold:
                match_found = True
                match_score = float(fuzzy_score)

        if match_found:
            # Get document metadata
            doc_data = documents_dict.get(doc_id, {})
            country_iso = doc_data.get('iso', '')

            # Get country name from countries_dict
            country_info = countries_dict.get(country_iso, {})
            country_name = country_info.get('country', doc_data.get('country', 'Unknown'))

            # Get context if requested
            context_before, context_after = get_context_window(
                segment_id, model_dict, config.context_window
            )

            result = SearchResult(
                segment_id=segment_id,
                text=segment_text,
                score=match_score,
                document_id=doc_id,
                document_name=doc_data.get('document', doc_data.get('name', '')),
                country=country_name,
                country_iso=country_iso,
                year=int(doc_data.get('year', 0)) if str(doc_data.get('year', '')).isdigit() else 0,
                doc_type=doc_data.get('type', ''),
                context_before=context_before,
                context_after=context_after
            )

            # Enrich with metadata
            result = enrich_result_with_metadata(result)
            results.append(result)

            # Respect max results limit if set
            if config.max_results_per_topic is not None and len(results) >= config.max_results_per_topic:
                print(f'⚠️  Reached maximum results limit ({config.max_results_per_topic})')
                break

    print(f'✓ Found {len(results)} matches')

    # Perform clustering if we have multiple results
    cluster_info_dict = {}
    if len(results) > 1:
        try:
            cluster_info_dict = cluster_search_results_internal(
                results, model_dict, config.cluster_threshold, search_query=keyword
            )
        except Exception as e:
            print(f'⚠️  Clustering error: {str(e)}')
            # Fallback: all singletons
            cluster_info_dict = {
                'cluster_dict': {'singletons': list(range(len(results)))},
                'num_clusters': 0,
                'num_singletons': len(results),
                'cluster_names': {'singletons': 'Unclustered'}
            }
            for result in results:
                result.cluster_id = 'singletons'
                result.cluster_name = 'Unclustered'
    elif len(results) == 1:
        # Single result handling
        cluster_info_dict = {
            'cluster_dict': {'singletons': [0]},
            'num_clusters': 0,
            'num_singletons': 1,
            'cluster_names': {'singletons': 'Unclustered'}
        }
        results[0].cluster_id = 'singletons'
        results[0].cluster_name = 'Unclustered'

    return results, cluster_info_dict

# Batch search function
def run_metadata_batch_search(config: SearchConfig, model_dict, encoder):
    """Run batch search with metadata filtering (supports both semantic and keyword search)."""
    from datetime import datetime

    start_time = datetime.now()
    all_results = {}
    all_cluster_info = {}
    total_results = 0

    search_mode = 'Boolean' if config.boolean_mode else ('Keyword' if config.keyword_search else 'Semantic')
    print(f'\n🚀 Starting batch {search_mode.lower()} search...')
    print(f'📝 Topics: {len(config.topics)}')
    print(f'🔍 Filters: {config.filters.get_summary()}')
    print()

    for topic in tqdm(config.topics, desc='Searching topics'):
        try:
            if config.boolean_mode:
                # Boolean search mode
                results, cluster_info = run_boolean_search(topic, config, model_dict)
            elif config.keyword_search:
                # Keyword search mode
                results, cluster_info = run_keyword_search(topic, config, model_dict)
            else:
                # Semantic search mode
                results, cluster_info = run_similarity_search(topic, config, model_dict, encoder)

            all_results[topic] = results
            all_cluster_info[topic] = cluster_info
            total_results += len(results)

            if results:
                print(f'  ✓ {topic}: {len(results)} results ({cluster_info.get("num_clusters", 0)} clusters, {cluster_info.get("num_singletons", 0)} singletons)')
            else:
                print(f'  ⚠️  {topic}: No results')
        except Exception as e:
            print(f'  ❌ {topic}: Error - {e}')
            import traceback
            traceback.print_exc()
            all_results[topic] = []
            all_cluster_info[topic] = {}

    end_time = datetime.now()
    execution_time = (end_time - start_time).total_seconds()

    print(f'\n✅ Search completed!')
    print(f'⏱️  Execution time: {execution_time:.2f} seconds')
    print(f'📊 Total results: {total_results}')

    return all_results, all_cluster_info

# Export function
def export_search_results(all_results: Dict[str, List[SearchResult]], all_cluster_info: Dict,
                         config: SearchConfig, output_dir: str = 'outputs', enable_export: bool = True, iterative_names: bool = False):
    """Export search results to CSV with metadata, context, and cluster names."""
    if not enable_export:
        return

    from pathlib import Path
    import csv
    from datetime import datetime

    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    search_type = 'Boolean Search' if config.boolean_mode else ('Keyword' if config.keyword_search else 'Semantic')

    for topic, results in all_results.items():
        if not results:
            continue

        # Get cluster info
        cluster_info = all_cluster_info.get(topic, {})
        num_clusters = cluster_info.get('num_clusters', 0)
        num_singletons = cluster_info.get('num_singletons', len(results))

        # Create safe filename with keyword suffix if applicable
        safe_topic = ''.join(c if c.isalnum() or c in (' ', '_') else '_' for c in topic).strip().replace(' ', '_')[:50]

        # Clean filename: remove leading underscores and collapse multiple underscores
        safe_topic = safe_topic.lstrip('_')
        safe_topic = re.sub(r'_+', '_', safe_topic)
        safe_topic = safe_topic.rstrip('_')

        if config.keyword_search:
            filename = f'{safe_topic}_keyword.csv'
        else:
            filename = f'{safe_topic}.csv'

        # Add timestamp if iterative names enabled
        if iterative_names:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            name_part = filename.rsplit('.', 1)[0]
            filename = f'{name_part}_{timestamp}.csv'

        filepath = output_path / filename

        # Build CSV with context and cluster name columns
        search_threshold_display = 'n/a' if config.keyword_search else config.search_threshold
        cluster_threshold_display = config.cluster_threshold
        csv_rows = [
            ['Topic', topic],
            ['Search type', search_type],
            ['Search threshold', search_threshold_display],
            ['Cluster threshold', cluster_threshold_display],
            ['Number of search results', len(results)],
            ['Number of clusters', num_clusters + 1],  # +1 to include singletons group
            ['Context window', config.context_window],
            ['Active filters', config.filters.get_summary()],
            [''],
            ['Cluster ID', 'Cluster Name', 'Search score', 'Segment text', 'Context before', 'Context after', 'Segment ID',
             'Document name', 'Document year', 'Country', 'Country ISO', 'Region', 'Subregion',
             'Income Group', 'Freedom Status', 'Accept result', 'Notes']
        ]

        # Add data rows with context and cluster names
        for i, result in enumerate(results):
            cluster_id = result.cluster_id if result.cluster_id is not None else 'singletons'
            cluster_name = result.cluster_name if result.cluster_name else str(cluster_id)
            csv_rows.append([
                str(cluster_id),
                cluster_name,
                f'{result.score:.4f}',
                result.text,
                result.context_before,
                result.context_after,
                result.segment_id,
                result.document_name,
                result.year,
                result.country,
                result.country_iso,
                result.region,
                result.subregion,
                result.income_group,
                result.freedom_status,
                'Yes',
                ''
            ])

        # Write CSV with UTF-8 BOM
        with open(filepath, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f)
            writer.writerows(csv_rows)

        print(f'  ✓ Exported: {filename} ({len(results)} results, {num_clusters} clusters + {num_singletons} singletons)')

# Display function
def display_batch_results(all_results: Dict[str, List[SearchResult]], all_cluster_info: Dict,
                         config: SearchConfig, max_per_topic: int = 5):
    """Display batch search results with cluster names and optional context."""
    from collections import defaultdict

    for topic, results in all_results.items():
        topic_safe = escape(topic)
        if not results:
            display(HTML(f'''
            <div style="background-color: #ffebee; padding: 15px; margin: 20px 0;
                        border-radius: 8px; border-left: 4px solid #f44336;">
                <strong>❌ No results found for: "{topic_safe}"</strong>
            </div>
            '''))
            continue

        cluster_info = all_cluster_info.get(topic, {})
        num_clusters = cluster_info.get('num_clusters', 0)
        num_singletons = cluster_info.get('num_singletons', 0)

        # Header
        search_type = 'Boolean Search' if config.boolean_mode else ('Keyword Search' if config.keyword_search else 'Semantic Search')
        display(HTML(f'''
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    padding: 20px; border-radius: 10px; color: white; margin: 20px 0;">
            <h2 style="margin: 0; font-size: 24px;">🔍 {escape(search_type)} Results: "{topic_safe}"</h2>
            <p style="margin: 10px 0 0 0; opacity: 0.9;">
                Found {len(results)} results |
                {num_clusters} clusters + {num_singletons} singletons
            </p>
        </div>
        '''))

        # Group by document
        by_document = defaultdict(list)
        for result in results[:max_per_topic * 5]:  # Show more results
            by_document[result.document_id].append(result)

        # Display each document group
        for doc_id, doc_results in list(by_document.items())[:max_per_topic]:
            first_result = doc_results[0]

            # Document header with metadata
            metadata_tags = []
            if first_result.region:
                metadata_tags.append(f'🌍 {escape(first_result.region)}')
            if first_result.subregion:
                metadata_tags.append(f'📍 {escape(first_result.subregion)}')
            if first_result.income_group:
                metadata_tags.append(f'💰 {escape(first_result.income_group)}')
            if first_result.freedom_status:
                metadata_tags.append(f'🗳️ {escape(first_result.freedom_status)}')

            metadata_str = ' | '.join(metadata_tags)

            display(HTML(f'''
            <div style="background-color: #f5f5f5; padding: 15px; margin: 15px 0;
                        border-radius: 8px; border-left: 5px solid #4CAF50;">
                <h3 style="margin: 0 0 10px 0; color: #546e7a;">
                    📄 {escape(first_result.country)} ({first_result.year})
                </h3>
                <p style="margin: 5px 0; color: #555; font-size: 14px;">
                    <strong>{escape(first_result.document_name)}</strong>
                </p>
                <p style="margin: 5px 0; color: #777; font-size: 12px;">
                    {metadata_str}
                </p>
                <p style="margin: 5px 0; color: #999; font-size: 12px;">
                    📊 {len(doc_results)} results in this document
                </p>
            </div>
            '''))

            # Display individual results
            for i, result in enumerate(doc_results[:3], 1):
                score_indicator = '🎯' if result.score > 0.8 else '✅' if result.score > 0.65 else '📍'

                # Display cluster name if available, otherwise cluster ID
                if result.cluster_name and result.cluster_name != 'Unclustered':
                    cluster_display = f' | 🏷️ {escape(result.cluster_name)}'
                elif result.cluster_id is not None and result.cluster_id != 'singletons':
                    cluster_display = f' | Cluster {escape(str(result.cluster_id))}'
                else:
                    cluster_display = ''

                # Build context HTML if context is available
                context_html = ''
                if result.context_before or result.context_after:
                    context_html = '<div style="background-color: #f9f9f9; padding: 10px; margin: 8px 0; border-left: 3px solid #ccc; font-size: 12px; color: #666; font-style: italic;">'
                    if result.context_before:
                        cb = escape(result.context_before[:300]) + ('...' if len(result.context_before) > 300 else '')
                        context_html += f'<p style="margin: 0 0 5px 0;"><strong>↑ Before:</strong> {cb}</p>'
                    if result.context_after:
                        ca = escape(result.context_after[:300]) + ('...' if len(result.context_after) > 300 else '')
                        context_html += f'<p style="margin: 5px 0 0 0;"><strong>↓ After:</strong> {ca}</p>'
                    context_html += '</div>'

                result_text = escape(result.text[:500]) + ('...' if len(result.text) > 500 else '')
                display(HTML(f'''
                <div style="background-color: white; padding: 12px; margin: 8px 0 8px 20px;
                            border-radius: 5px; border-left: 3px solid #9C27B0;">
                    <p style="margin: 0 0 5px 0; color: #888; font-size: 11px;">
                        {score_indicator} Score: {result.score:.3f}{cluster_display}
                    </p>
                    <p style="margin: 0 0 8px 0; color: #333; line-height: 1.6;">
                        {result_text}
                    </p>
                    {context_html}
                </div>
                '''))

            if len(doc_results) > 3:
                display(HTML(f'''
                <p style="margin: 5px 0 5px 20px; color: #999; font-size: 12px; font-style: italic;">
                    ... and {len(doc_results) - 3} more results in this document
                </p>
                '''))

        if len(by_document) > max_per_topic:
            display(HTML(f'''
            <p style="margin: 10px 0; color: #666; font-size: 12px; font-style: italic;">
                ... showing {max_per_topic} of {len(by_document)} documents with results (see exported CSV for all)
            </p>
            '''))

print('✅ Search execution functions defined (semantic + keyword search with cluster naming)')

In [ ]:
%%capture
# Boolean search implementation
from _library.boolean_parser import parse_boolean_query, BooleanQueryParser
from _library.boolean_executor import execute_boolean_query, BooleanQueryExecutor

def run_keyword_search_internal(keyword: str, model_dict: dict) -> List[Tuple[str, float]]:
    """
    Internal keyword search function for Boolean executor.

    Performs basic keyword pattern matching without metadata filtering or clustering.
    Returns raw list of (segment_id, score) tuples.

    Args:
        keyword: Search term (may include wildcards * and ?)
        model_dict: Data model containing segments

    Returns:
        List of (segment_id, score) tuples
    """
    segments_dict = model_dict['segments_dict']

    # Escape special regex chars except * and ?
    import re
    pattern = re.escape(keyword)
    # Replace escaped wildcards with regex equivalents
    pattern = pattern.replace(r'\*', '.*').replace(r'\?', '.')
    # Add word boundaries
    pattern = r'\b' + pattern + r'\b'

    # Compile pattern (case-insensitive)
    try:
        regex = re.compile(pattern, re.IGNORECASE)
    except re.error as e:
        print(f'⚠️  Invalid regex pattern for term "{keyword}": {e}')
        return []

    # Search all segments
    results = []
    for segment_id, segment_data in segments_dict.items():
        text = get_segment_text_safe(segment_data)
        if not text:
            continue
        if regex.search(text):
            # Score is 100.0 for keyword matches (normalized to 0-100 range)
            results.append((segment_id, 100.0))

    return results


def run_boolean_search(query: str, config: SearchConfig, model_dict) -> Tuple[List[SearchResult], Dict]:
    """
    Run Boolean search with metadata filtering and clustering.

    Supports Boolean operators (AND, OR, NOT) with parentheses grouping.

    Args:
        query: Boolean query string (e.g., "cyber AND attack NOT terrorism")
        config: Search configuration
        model_dict: Data model containing segments and metadata

    Returns:
        Tuple of (results, cluster_info_dict)

    Example queries:
        - "cyber AND attack"
        - "(nuclear OR radiological) AND weapon"
        - "terrorism AND NOT domestic"
        - "(cyber* OR digital) AND threat AND NOT terrorism"
    """
    print(f'🔍 Boolean search: {query}')

    # Get filtered documents
    doc_df = METADATA['documents']
    all_doc_ids = doc_df['File prefix'].astype(str).tolist()
    filtered_doc_ids = apply_metadata_filters(all_doc_ids, config.filters)

    if not filtered_doc_ids:
        print('⚠️  No documents match the filter criteria')
        return [], {}

    print(f'📄 Searching {len(filtered_doc_ids)} documents')

    # Parse Boolean query into AST
    try:
        parser = BooleanQueryParser()
        ast = parser.parse(query)
        print(f'✓ Query parsed successfully')
    except Exception as e:
        print(f'❌ Query parse error: {e}')
        return [], {}

    # Create filtered model_dict with only selected documents
    segments_dict = model_dict['segments_dict']
    filtered_segments_dict = {
        seg_id: seg_data
        for seg_id, seg_data in segments_dict.items()
        if seg_id.split('/')[0] in filtered_doc_ids
    }

    filtered_model_dict = model_dict.copy()
    filtered_model_dict['segments_dict'] = filtered_segments_dict

    # Execute Boolean query
    try:
        executor = BooleanQueryExecutor(
            model_dict=filtered_model_dict,
            keyword_search_func=run_keyword_search_internal,
            scoring_strategy=config.boolean_scoring
        )
        results = executor.execute(ast)
        print(f'✓ Found {len(results)} matching segments')
    except Exception as e:
        print(f'❌ Query execution error: {e}')
        return [], {}

    if not results:
        return [], {}

    # Apply max results limit if set
    if config.max_results_per_topic is not None:
        results = results[:config.max_results_per_topic]

    # Create SearchResult objects
    documents_dict = model_dict['documents_dict']
    countries_dict = model_dict['countries_dict']

    search_results = []
    for segment_id, score in results:
        doc_id = segment_id.split('/')[0]

        # Get segment text
        segment_data = segments_dict.get(segment_id, {})
        text = get_segment_text_safe(segment_data) or ''

        # Get document metadata
        doc_data = documents_dict.get(doc_id, {})
        doc_name = doc_data.get('document', doc_data.get('name', ''))
        year = doc_data.get('year', 0)
        doc_type = doc_data.get('type', 'Unknown')

        # Get country information (matching keyword search pattern)
        country_iso = doc_data.get('iso', '')
        country_info = countries_dict.get(country_iso, {})
        country_name = country_info.get('country', doc_data.get('country', 'Unknown'))

        # Get context windows if configured
        context_before = ''
        context_after = ''
        if config.context_window > 0:
            context_before, context_after = get_context_window(
                segment_id, model_dict, config.context_window
            )

        # Create search result
        result = SearchResult(
            segment_id=segment_id,
            text=text,
            score=score,
            document_id=doc_id,
            document_name=doc_name,
            country=country_name,
            country_iso=country_iso,
            year=year,
            doc_type=doc_type,
            context_before=context_before,
            context_after=context_after
        )

        # Enrich with metadata
        result = enrich_result_with_metadata(result)

        search_results.append(result)

    # Cluster results
    try:
        cluster_info = cluster_search_results_internal(
            search_results,
            model_dict,
            cluster_threshold=config.cluster_threshold,
            search_query=query
        )

        # Cluster names already assigned by cluster_search_results_internal
        print(f'✓ Clustered into {cluster_info.get("num_clusters", 0)} clusters '
              f'({cluster_info.get("num_singletons", 0)} singletons)')

    except Exception as e:
        print(f'⚠️  Clustering failed: {e}')
        print('Continuing with all results as singletons')
        cluster_info = {
            'cluster_dict': {},
            'cluster_labels': [-1] * len(search_results),
            'num_clusters': 0,
            'num_singletons': len(search_results),
            'cluster_names': {}
        }

    return search_results, cluster_info

print('✅ Boolean search functions defined')

In [ ]:
_loading_widget = widgets.HTML(value="""
<div style=\"padding: 20px; font-family: sans-serif; font-size: 1.1em; color: #555;\">
    &#9203; <strong>Loading model files &mdash; this may take several minutes.</strong><br>
    <span style=\"font-size: 0.9em;\">Please leave this window open. The search interface will appear automatically when ready.</span>
</div>
""")
display(_loading_widget)


In [ ]:
%%capture
MODEL_DICT, ENCODER = initialize_model()


In [ ]:
_loading_widget.value = ''
interface = create_search_interface()
display(interface)
